In [3]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='./.env_example')
API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=API_KEY)

In [4]:
# Create a VERY long message to hit the limit
long_story = "Once upon a time, there was a programmer who loved Python. " * 100000

print(f"📏 Story length: {len(long_story):,} characters")
print(f"📏 Estimated tokens: ~{len(long_story) // 4:,}\n")

try:
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=long_story,
        config={"max_output_tokens": 100}
    )
    print("✅ Request succeeded!")
    print(f"📊 Tokens used: {response.usage_metadata.total_token_count:,}")
    print(f"\n💡 Gemini 2.5 Flash has a HUGE context window (~1M tokens)")
    print(f"   So this request fits comfortably!")
except Exception as e:
    print(f"❌ ERROR: {e}")
    print("\n💡 This is what happens when you exceed the context window!")

📏 Story length: 5,900,000 characters
📏 Estimated tokens: ~1,475,000

❌ ERROR: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 250000, model: gemini-2.5-flash-lite\nPlease retry in 17.696402228s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_input_token_count', 'quotaId': 'GenerateContentInputTokensPerMo

In [5]:
messages = []

def chat(user_message):
    """Send message and track tokens"""
    messages.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )
    
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=messages
    )
    
    messages.append(
        types.Content(role="model", parts=[types.Part(text=response.text)])
    )
    
    # Show token usage
    total_tokens = response.usage_metadata.total_token_count
    print(f"🤖: {response.text}")
    print(f"📊 Total tokens used: {total_tokens} (includes ALL {len(messages)} messages)\n")
    
    return response.text

# Start conversation
print("👤: Hi! My name is Yash")
chat("Hi! My name is Yash")

print("👤: I'm 25 years old")
chat("I'm 25 years old")

print("👤: I love Python programming")
chat("I love Python programming")

print("👤: I work as a software engineer")
chat("I work as a software engineer")

print("👤: What's my name?")
chat("What's my name?")

👤: Hi! My name is Yash
🤖: Hi Yash! It's nice to meet you. How can I help you today? 😊
📊 Total tokens used: 26 (includes ALL 2 messages)

👤: I'm 25 years old
🤖: Great! Thanks for letting me know, Yash.  That's a good age, full of possibilities.

Is there anything specific you'd like to talk about or ask me?
📊 Total tokens used: 74 (includes ALL 4 messages)

👤: I love Python programming
🤖: That's fantastic, Yash! Python is a really versatile and popular programming language. It's used for so many things, from web development and data science to AI and automation.

What do you enjoy most about Python? Are you working on any personal projects, or are you using it for work or studies? I'm curious to hear what draws you to it!
📊 Total tokens used: 156 (includes ALL 6 messages)

👤: I work as a software engineer
🤖: That's excellent, Yash! Working as a software engineer is a very rewarding and in-demand career.

And it's no surprise you love Python, given your profession. It's a staple for many

'Your name is Yash. 😊'

In [6]:
MAX_MESSAGES = 6  # Keep only last 6 messages (3 exchanges)

def chat_with_limit(user_message):
    """Chat with context window management"""
    messages.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )
    
    # Remove old messages if too many
    if len(messages) > MAX_MESSAGES:
        removed = messages.pop(0)  # Remove oldest
        print(f"🗑️  Removed old message: {removed.parts[0].text[:50]}...\n")
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=messages
    )
    
    messages.append(
        types.Content(role="model", parts=[types.Part(text=response.text)])
    )
    
    print(f"🤖: {response.text}")
    print(f"📊 Messages in memory: {len(messages)}\n")
    
    return response.text

# Reset and try again
messages = []

print("👤: My favorite color is blue")
chat_with_limit("My favorite color is blue")

print("👤: I have a dog named Max")
chat_with_limit("I have a dog named Max")

print("👤: I live in Mumbai")
chat_with_limit("I live in Mumbai")

print("👤: I enjoy hiking")
chat_with_limit("I enjoy hiking")

# This will forget the first message!
print("👤: What's my favorite color?")
chat_with_limit("What's my favorite color?")

👤: My favorite color is blue
🤖: Blue is a fantastic choice! It's such a versatile and often calming color.

What do you like most about blue? Do you have a favorite shade, like a deep navy, a sky blue, or a vibrant turquoise?
📊 Messages in memory: 2

👤: I have a dog named Max
🤖: Aw, Max! That's such a classic and wonderful dog name. I bet he's a fantastic companion.

What kind of dog is Max, and what's he like? Does he have any funny quirks or favorite activities?
📊 Messages in memory: 4

👤: I live in Mumbai
🤖: Oh, Mumbai! What an incredibly vibrant and dynamic city. It's known for its incredible energy, amazing food, and of course, Bollywood.

What do you enjoy most about living in Mumbai? Do you have a favorite spot, or perhaps a favorite local food experience?
📊 Messages in memory: 6

👤: I enjoy hiking
🗑️  Removed old message: My favorite color is blue...

🤖: That's fantastic! Living in Mumbai and enjoying hiking is a great combination, as you have some surprisingly green spaces wit

"That's a fun question, but I actually don't know your favorite color! You haven't mentioned it yet.\n\nWhat *is* your favorite color? I'm curious!"